# Figure 3: Evidence For Reference-Conditioned Modeling

- Figure / panels: `Fig3a`, `Fig3b`, `Fig3c`, `Fig3d`
- Inputs: `artifacts/results/<dataset>/metrics_nearest.csv`, `artifacts/results/<dataset>/metrics_random.csv`, Systema baseline outputs, payload `pkl`
- Outputs: `artifacts/paper_figures/main/Fig3_ReferenceConditioning/*`
- Run order: run after the Fig2 benchmark notebook; this notebook focuses on nearest vs random and reference baselines
- Dataset selector: edit `SELECTED_DATASET_KEYS` in the parameter cell below
- Role: Main text


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
from scripts.trishift.analysis import baseline_panel, stratified_benchmark

apply_gears_paper_style(font_scale=1.0)
from scripts.trishift.analysis._result_adapter import load_payload_item
from trishift._utils import normalize_condition
importlib.reload(baseline_panel)
importlib.reload(stratified_benchmark)


In [ ]:
ALL_DATASET_SPECS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit'},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson'},
    {'dataset_key': 'norman', 'dataset_label': 'Norman (single)'},
    {'dataset_key': 'replogle_k562_essential', 'dataset_label': 'Replogle K562'},
    {'dataset_key': 'replogle_rpe1_essential', 'dataset_label': 'Replogle RPE1'},
]
AVAILABLE_DATASET_KEYS = [spec['dataset_key'] for spec in ALL_DATASET_SPECS]
SELECTED_DATASET_KEYS = ['adamson', 'norman']  # edit here
invalid_dataset_keys = [key for key in SELECTED_DATASET_KEYS if key not in AVAILABLE_DATASET_KEYS]
if invalid_dataset_keys:
    raise ValueError(f'Unknown dataset keys: {invalid_dataset_keys}. Available: {AVAILABLE_DATASET_KEYS}')
DATASET_SPECS = [spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] in SELECTED_DATASET_KEYS]
if not DATASET_SPECS:
    raise ValueError('SELECTED_DATASET_KEYS produced an empty dataset list.')

REFERENCE_MODELS = ['trishift_nearest', 'trishift_random', 'systema_nonctl_mean', 'systema_matching_mean']  # edit here
DISTANCE_MODELS = ['trishift_nearest', 'trishift_random']  # edit here
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig3_ReferenceConditioning'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE_CASE_OVERRIDE = {'dataset_key': 'norman', 'condition': 'CITED1+ctrl', 'split_id': 4}  # set to None for auto
MODEL_COLORS = {
    'TriShift nearest': '#4C78A8',
    'TriShift random': '#F28E2B',
    'Systema nonctl-mean': '#9A8C98',
    'Systema matching-mean': '#7A6FF0',
    'Truth': '#7F7F7F',
}
print('Selected datasets:', SELECTED_DATASET_KEYS)
print('Selected splits:', SPLIT_IDS)
print('Reference case override:', REFERENCE_CASE_OVERRIDE)


In [ ]:
def safe_reference_benchmark(dataset_key: str) -> dict[str, object]:
    out_dir = OUT_ROOT / f'reference_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        return {'status': 'ready', 'result': baseline_panel.run_baseline_panel(dataset=dataset_key, models=REFERENCE_MODELS, split_ids=SPLIT_IDS, out_root=out_dir)}
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc)}


def safe_distance_summary(dataset_key: str) -> dict[str, object]:
    out_dir = OUT_ROOT / f'distance_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        return {'status': 'ready', 'result': stratified_benchmark.run_stratified_benchmark(dataset=dataset_key, models=DISTANCE_MODELS, split_ids=SPLIT_IDS, out_root=out_dir)}
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc)}


def _style_axis(ax: plt.Axes, *, grid_axis: str = 'y') -> None:
    paper_style_axis(ax, grid_axis=grid_axis)
    ax.set_axisbelow(True)


def render_grouped_bar(df: pd.DataFrame, metric: str, metric_label: str, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(9.5, 4.7), dpi=220)
    if df.empty:
        ax.text(0.5, 0.5, f'No rows for {metric}', ha='center', va='center'); ax.axis('off')
    else:
        sns.barplot(data=df, x='dataset_label', y=f'mean_{metric}', hue='label', palette=MODEL_COLORS, ax=ax, saturation=0.95)
        ax.set_xlabel('')
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.tick_params(axis='x', rotation=18)
        ax.legend(title='', frameon=False, ncol=min(4, df['label'].nunique()), loc='upper center', bbox_to_anchor=(0.5, 1.18))
        for patch in ax.patches:
            patch.set_edgecolor('white'); patch.set_linewidth(0.8)
        _style_axis(ax)
    fig.tight_layout(); fig.savefig(out_path, bbox_inches='tight'); plt.close(fig)


def _build_reference_case(dataset_key: str, condition: str, split_id: int):
    condition_key = normalize_condition(condition)
    try:
        _, near_payload = load_payload_item(dataset=dataset_key, model_name='trishift_nearest', split_id=split_id, condition=None)
        _, rand_payload = load_payload_item(dataset=dataset_key, model_name='trishift_random', split_id=split_id, condition=None)
    except Exception:
        return None, None
    near_item = {normalize_condition(k): v for k, v in near_payload.items()}.get(condition_key)
    rand_item = {normalize_condition(k): v for k, v in rand_payload.items()}.get(condition_key)
    if near_item is None or rand_item is None:
        return None, None
    truth = np.asarray(near_item['Truth'], dtype=float)
    ctrl = np.asarray(near_item['Ctrl'], dtype=float)
    near_pred = np.asarray(near_item['Pred'], dtype=float)
    rand_pred = np.asarray(rand_item['Pred'], dtype=float)
    genes = np.asarray(near_item.get('DE_name', [f'g{i}' for i in range(near_pred.shape[1])])).astype(str)
    truth_delta = truth.mean(axis=0) - ctrl.mean(axis=0)
    order = np.argsort(-np.abs(truth_delta))[:12]
    rows = [
        pd.DataFrame({'Gene': genes[order], 'Expression': truth_delta[order], 'Group': 'Truth'}),
        pd.DataFrame({'Gene': genes[order], 'Expression': near_pred.mean(axis=0)[order] - ctrl.mean(axis=0)[order], 'Group': 'TriShift nearest'}),
        pd.DataFrame({'Gene': genes[order], 'Expression': rand_pred.mean(axis=0)[order] - ctrl.mean(axis=0)[order], 'Group': 'TriShift random'}),
    ]
    return {'dataset_key': dataset_key, 'condition': condition_key, 'split_id': int(split_id)}, pd.concat(rows, ignore_index=True)


def pick_reference_case():
    if REFERENCE_CASE_OVERRIDE is not None:
        case_meta, case_df = _build_reference_case(
            dataset_key=REFERENCE_CASE_OVERRIDE['dataset_key'],
            condition=REFERENCE_CASE_OVERRIDE['condition'],
            split_id=int(REFERENCE_CASE_OVERRIDE['split_id']),
        )
        if case_meta is not None and case_df is not None:
            case_meta['score_gain'] = np.nan
            case_meta['override'] = True
            return case_meta, case_df
    preferred_order = ['dixit', 'adamson', 'replogle_k562_essential', 'replogle_rpe1_essential', 'norman']
    for dataset_key in preferred_order:
        nearest_path = repo_root / 'artifacts' / 'results' / dataset_key / 'metrics_nearest.csv'
        random_path = repo_root / 'artifacts' / 'results' / dataset_key / 'metrics_random.csv'
        if not nearest_path.exists() or not random_path.exists():
            continue
        nearest_df = pd.read_csv(nearest_path)
        random_df = pd.read_csv(random_path)
        metric_cols = ['condition', 'split_id', 'pearson', 'deg_mean_r2', 'systema_corr_20de_allpert']
        common = nearest_df[metric_cols].merge(random_df[metric_cols], on=['condition', 'split_id'], suffixes=('_nearest', '_random'))
        if common.empty:
            continue
        common['score_gain'] = (common['pearson_nearest'] - common['pearson_random']).fillna(0.0) + (common['deg_mean_r2_nearest'] - common['deg_mean_r2_random']).fillna(0.0) + (common['systema_corr_20de_allpert_nearest'] - common['systema_corr_20de_allpert_random']).fillna(0.0)
        best = common.sort_values('score_gain', ascending=False).iloc[0]
        case_meta, case_df = _build_reference_case(dataset_key=dataset_key, condition=best['condition'], split_id=int(best['split_id']))
        if case_meta is None or case_df is None:
            continue
        case_meta['score_gain'] = float(best['score_gain'])
        case_meta['override'] = False
        return case_meta, case_df
    return None, None


In [ ]:
reference_runs, distance_runs = [], []
for spec in DATASET_SPECS:
    reference_runs.append({**spec, **safe_reference_benchmark(spec['dataset_key'])})
    distance_runs.append({**spec, **safe_distance_summary(spec['dataset_key'])})

reference_frames = []
for row in reference_runs:
    if row['status'] == 'ready':
        df = row['result']['summary_df'].copy(); df['dataset_label'] = row['dataset_label']; reference_frames.append(df)
reference_summary_df = pd.concat(reference_frames, ignore_index=True) if reference_frames else pd.DataFrame()
reference_summary_df.to_csv(OUT_ROOT / 'reference_summary_all.csv', index=False, encoding='utf-8-sig')
display(reference_summary_df.head())


In [ ]:
nearest_random_df = reference_summary_df[reference_summary_df['model_name'].isin(['trishift_nearest', 'trishift_random'])].copy()
render_grouped_bar(nearest_random_df, 'pearson', 'Fig3a: TriShift nearest vs random (Pearson)', OUT_ROOT / 'fig3a_nearest_vs_random.png')

signal_df = reference_summary_df[reference_summary_df['model_name'].isin(REFERENCE_MODELS)].copy()
metric_cols = ['mean_systema_corr_20de_allpert', 'mean_systema_corr_deg_r2', 'mean_pearson']
if not signal_df.empty:
    ref_heatmap = signal_df[['label', 'dataset_label'] + [c for c in metric_cols if c in signal_df.columns]].copy().melt(id_vars=['label', 'dataset_label'], var_name='metric', value_name='value')
    ref_heatmap['metric'] = ref_heatmap['metric'].str.replace('mean_', '', regex=False)
    ref_heatmap['column'] = ref_heatmap['dataset_label'] + '\n' + ref_heatmap['metric']
    ref_table = ref_heatmap.pivot_table(index='label', columns='column', values='value', aggfunc='mean')
else:
    ref_table = pd.DataFrame()
fig, ax = plt.subplots(figsize=(max(9, ref_table.shape[1] * 1.0 if not ref_table.empty else 9), 4.8), dpi=220)
if ref_table.empty:
    ax.text(0.5, 0.5, 'No reference-baseline summary available', ha='center', va='center'); ax.axis('off')
else:
    sns.heatmap(ref_table, annot=True, fmt='.3f', cmap='RdYlGn', linewidths=0.5, ax=ax)
    ax.set_title('Fig3b: TriShift vs reference baselines')
plt.tight_layout(); plt.savefig(OUT_ROOT / 'fig3b_reference_baselines.png', bbox_inches='tight'); plt.close()


In [ ]:
distance_frames = []
for row in distance_runs:
    if row['status'] != 'ready':
        continue
    summary_df = row['result']['summary_df'].copy()
    if 'stratum_name' not in summary_df.columns:
        continue
    sub = summary_df[summary_df['stratum_name'].astype(str) == 'train_distance_bin'].copy()
    if sub.empty:
        continue
    sub['dataset_label'] = row['dataset_label']; distance_frames.append(sub)
distance_summary_df = pd.concat(distance_frames, ignore_index=True) if distance_frames else pd.DataFrame()
distance_summary_df.to_csv(OUT_ROOT / 'distance_summary_all.csv', index=False, encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(8.8, 4.6), dpi=220)
required_distance_cols = {'model_name', 'dataset_label', 'stratum_value', 'pearson'}
if distance_summary_df.empty or not required_distance_cols.issubset(distance_summary_df.columns):
    ax.text(0.5, 0.5, 'No train-distance summary available', ha='center', va='center'); ax.axis('off')
else:
    work = distance_summary_df[distance_summary_df['model_name'].isin(['trishift_nearest', 'trishift_random'])].copy()
    label_map = {'trishift_nearest': 'TriShift nearest', 'trishift_random': 'TriShift random'}
    work['label'] = work['model_name'].map(label_map).fillna(work['model_name'])
    if set(work['stratum_value'].dropna().astype(str).unique()).issubset({'near', 'medium', 'far'}):
        work['stratum_value'] = pd.Categorical(work['stratum_value'], categories=['near', 'medium', 'far'], ordered=True)
    sns.lineplot(data=work, x='stratum_value', y='pearson', hue='label', style='dataset_label', markers=True, dashes=True, palette={'TriShift nearest': MODEL_COLORS['TriShift nearest'], 'TriShift random': MODEL_COLORS['TriShift random']}, linewidth=2.0, ax=ax)
    ax.set_xlabel('Train-distance bin')
    ax.set_ylabel('Mean Pearson')
    ax.set_title('Reference retrieval across train-distance bins')
    _style_axis(ax)
    ax.legend(title='', frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.24), ncol=2)
plt.tight_layout(); plt.savefig(OUT_ROOT / 'fig3c_distance_summary.png', bbox_inches='tight'); plt.close()
display(distance_summary_df.head())


In [ ]:
case_meta, case_df = pick_reference_case()
if case_meta is not None and case_df is not None:
    case_df.to_csv(OUT_ROOT / 'fig3d_reference_case_values.csv', index=False, encoding='utf-8-sig')
    plt.figure(figsize=(12.8, 5.1), dpi=220)
    ax = sns.barplot(data=case_df, x='Gene', y='Expression', hue='Group', palette=MODEL_COLORS, errorbar=None, saturation=0.95)
    for patch in ax.patches:
        patch.set_edgecolor('white'); patch.set_linewidth(0.7)
    ax.set_xlabel('')
    ax.set_ylabel('Expression change over control')
    ax.set_title(f"Reference-sensitive case: {case_meta['condition']} (split {case_meta['split_id']})")
    ax.tick_params(axis='x', rotation=32)
    ax.legend(title='', frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=min(3, case_df['Group'].nunique()))
    ax.axhline(y=0, color='#4A4A4A', linewidth=0.8)
    _style_axis(ax)
    plt.tight_layout(); plt.savefig(OUT_ROOT / 'fig3d_reference_case.png', bbox_inches='tight'); plt.close()
    print(case_meta); display(case_df.head(20))
else:
    print('No reference-sensitive case could be selected from current results.')
print(OUT_ROOT)
